will start by cleaning up the ACRG dataset, which i will use as a training set

In [31]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as pls

horizon_months = 24
out_dir = Path("data")
out_dir.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160)

acrg = pd.read_csv("acrg_clinical_full.csv")
need = ["patient", "sex", "age", "t_stage", "m_stage", "positive_nodes", "who_simple", "recurrence_event", "dfs_months", "outcome_inconsistent"]

assert not [c for c in need if c not in acrg.columns], "there are missing columns in the ACRG csv"
assert acrg["patient"].is_unique
print(acrg.shape)

(300, 60)


In [32]:
df = pd.DataFrame({
    "patient_id": acrg["patient"].astype(str),
    "age": acrg["age"],
    "sex": acrg["sex"].map({"M": "Male", "F": "Female"}),
    "serosa":(acrg["t_stage"] >= 3).astype(float).where(acrg["t_stage"].notna()),
    "n_level": pd.cut(acrg["positive_nodes"], [-1, 0, 6, np.inf], labels=[0, 1, 2]).astype(float),
    "grade_g3": acrg["who_simple"].map({"p/d": 1.0, "signet": 1.0, "w/d~m/d": 0.0}),
    "m1": (acrg["m_stage"] == 1).astype(float).where(acrg["m_stage"].notna()),
    "event": acrg["recurrence_event"],
    "time_months": acrg["dfs_months"],
    "location": acrg["location_code"].map({"antrum": "antrum", "body": "body", "cardia": "cardia", "whole": "other"}),
    "outcome_inconsistent": acrg["outcome_inconsistent"].astype(bool),
})
df["outcome_known"] = df["event"].notna() & df["time_months"].notna()
df.head()

,patient_id,age,sex,serosa,n_level,grade_g3,m1,event,time_months,location,outcome_inconsistent,outcome_known
0,107,67,Male,0.0,1.0,0.0,0.0,NaN,3.966667,body,False,False
1,108,68,Female,0.0,1.0,1.0,0.0,NaN,4.033333,body,False,False
2,109,42,Female,0.0,1.0,1.0,0.0,0.0,74.966667,antrum,False,True
3,160,69,Male,0.0,1.0,1.0,0.0,0.0,89.766667,antrum,False,True
4,164,68,Male,1.0,2.0,1.0,0.0,0.0,84.600000,antrum,False,True


In [33]:
core = ["age", "sex", "serosa", "n_level", "event", "time_months"]

steps = [
    ("all patients", pd.Series(True, index = df.index)),
    ("outcome is known", df["outcome_known"]),
    ("no distant metastasis", df["m1"].eq(0)),
    ("core variables are complete", df[core].notna().all(axis = 1)),
    ("no contradictory outcomes are recorded", ~df["outcome_inconsistent"]),
    ("follow-up time > 0", df["time_months"] > 0),
]

mask, rows = pd.Series(True, index = df.index), []
for label, m in steps:
    mask = mask & m.fillna(False)
    rows.append((label, int(mask.sum()), int(df.loc[mask, "event"].sum())))

flow = pd.DataFrame(rows, columns = ["step", "patients", "events"])
df["main_analysis"] = mask
flow

,step,patients,events
0,all patients,300,125
1,outcome is known,282,125
2,no distant metastasis,257,105
3,core variables are complete,255,105
4,no contradictory outcomes are recorded,252,103
5,follow-up time > 0,252,103


In [34]:
out_cols = ["patient_id", "age", "sex", "serosa", "n_level", "grade_g3", "event", "time_months", "location"]
final = df.loc[df["main_analysis"], out_cols].reset_index(drop = True)

print("Percent missing in the final set:"); display((final.isna().mean() * 100).round(1))
problems = []
if not final["age"].between(18, 100).all():   problems.append("age outside 18-100")
if not (final["time_months"] > 0).all():      problems.append("non-positive time")
if not final["event"].isin([0, 1]).all():     problems.append("event not 0/1")
if not final["patient_id"].is_unique:         problems.append("duplicate ids")
print("Checks:", "all passed" if not problems else problems)

Percent missing in the final set:


patient_id      0.0
age             0.0
sex             0.0
serosa          0.0
n_level         0.0
grade_g3       12.7
event           0.0
time_months     0.0
location        0.0
dtype: float64

Checks: all passed


In [35]:
final.to_csv(out_dir / "acrg_clean.csv", index=False)
flow.to_csv(out_dir / "acrg_flowchart.csv", index=False)
print("saved", final.shape)

saved (252, 9)


next will clean up my TCGA cohort

In [36]:
raw = pd.read_csv("stad_tcga_pan_can_atlas_2018_clinical_data.tsv", sep = "\t", low_memory = False)
cols = {
    "Patient ID": "patient_id", "Diagnosis Age": "age", "Sex": "sex", "Race Category": "race",
    "American Joint Committee on Cancer Publication Version Type": "ajcc_edition",
    "American Joint Committee on Cancer Tumor Stage Code": "t_code",
    "Neoplasm Disease Lymph Node Stage American Joint Committee on Cancer Code": "n_code",
    "American Joint Committee on Cancer Metastasis Stage Code": "m_code",
    "Neoplasm Histologic Grade": "grade",
    "Progression Free Status": "pfs_status",
    "Progress Free Survival (Months)": "pfs_months",
    "International Classification of Diseases for Oncology, Third Edition ICD-O-3 Site Code": "site_code",
}

assert not [k for k in cols if k not in raw.columns], "TCGA columns are not found"
t = raw[list(cols)].rename(columns=cols).copy()
for c in ["t_code", "n_code", "m_code", "ajcc_edition"]:
    t[c] = t[c].astype("string").str.upper().str.strip()
    assert t["patient_id"].is_unique
print(t.shape)

(440, 12)


In [37]:
def serosa_level(code, is7):
    if pd.isna(code) or code == "TX":
        return np.nan
    if code.startswith("T4"):
        return 1.0
    if code.startswith("T3"):
        return 0.0 if is7 else 1.0
    return 0.0

def node_level(code, is7):
    if pd.isna(code) or code == "NX":
        return np.nan
    if code == "N0":
        return 0.0
    if code == "N1":
        return 1.0
    if code == "N2":
        return 1.0 if is7 else 2.0
    return 2.0

In [38]:
is7 = t["ajcc_edition"].eq("7TH").fillna(False)
tcga_df = pd.DataFrame({
    "patient_id": t["patient_id"],
    "age": t["age"],
    "sex": t["sex"],
    "serosa": [serosa_level(c, e) for c, e in zip(t["t_code"], is7)],
    "n_level": [node_level(c, e) for c, e in zip(t["n_code"], is7)],
    "grade_g3": t["grade"].map({"G3": 1.0, "G2": 0.0, "G1": 0.0}),
    "m1": t["m_code"].map({"M0": 0.0, "M1": 1.0}),            # MX stays missing
    "event": t["pfs_status"].astype("string").str.startswith("1").astype(float),
    "time_months": t["pfs_months"],
    "location": t["site_code"].map({"C16.0": "cardia", "C16.1": "body", "C16.2": "body", "C16.3": "antrum"}).fillna("other"),
})
tcga_df["outcome_known"] = tcga_df["event"].notna() & tcga_df["time_months"].notna()
tcga_df.head()

,patient_id,age,sex,serosa,n_level,grade_g3,m1,event,time_months,location,outcome_known
0,TCGA-3M-AB46,70.0,Male,0.0,0.0,0.0,NaN,0.0,58.026761,other,True
1,TCGA-3M-AB47,51.0,Male,1.0,2.0,1.0,NaN,1.0,12.986159,other,True
2,TCGA-B7-5816,51.0,Female,1.0,0.0,1.0,0.0,0.0,26.695598,body,True
3,TCGA-B7-5818,62.0,Male,0.0,0.0,1.0,0.0,0.0,11.703981,body,True
4,TCGA-B7-A5TI,52.0,Male,1.0,2.0,1.0,0.0,0.0,19.561429,body,True


In [39]:
CORE = ["age", "sex", "serosa", "n_level", "event", "time_months"]
steps = [
    ("All patients",                  pd.Series(True, index=tcga_df.index)),
    ("Outcome known",                 tcga_df["outcome_known"]),
    ("No distant metastasis (M0)",    tcga_df["m1"].eq(0)),
    ("Core variables complete",       tcga_df[CORE].notna().all(axis=1)),
    ("Follow-up time > 0",            tcga_df["time_months"] > 0),
]
mask, rows = pd.Series(True, index=tcga_df.index), []
for label, m in steps:
    mask = mask & m.fillna(False)
    rows.append((label, int(mask.sum()), int(tcga_df.loc[mask, "event"].sum())))
tcga_flow = pd.DataFrame(rows, columns=["step", "patients", "events"])
tcga_df["main_analysis"] = mask
tcga_flow

,step,patients,events
0,All patients,440,142
1,Outcome known,437,142
2,No distant metastasis (M0),388,126
3,Core variables complete,366,125
4,Follow-up time > 0,359,125


In [40]:
OUT_COLS = ["patient_id", "age", "sex", "serosa", "n_level", "grade_g3", "location", "event", "time_months"]
tcga_final = tcga_df.loc[tcga_df["main_analysis"], OUT_COLS].reset_index(drop=True)

print("Percent missing:"); display((tcga_final.isna().mean() * 100).round(1))
problems = []
if not tcga_final["age"].between(18, 100).all():   problems.append("age outside 18-100")
if not (tcga_final["time_months"] > 0).all():      problems.append("non-positive time")
if not tcga_final["event"].isin([0, 1]).all():     problems.append("event not 0/1")
if not tcga_final["patient_id"].is_unique:         problems.append("duplicate ids")
print("Checks:", "all passed" if not problems else problems)

Percent missing:


patient_id     0.0
age            0.0
sex            0.0
serosa         0.0
n_level        0.0
grade_g3       1.7
location       0.0
event          0.0
time_months    0.0
dtype: float64

Checks: all passed


In [41]:
tcga_final.to_csv(out_dir / "tcga_clean.csv", index=False)
tcga_flow.to_csv(out_dir / "tcga_flowchart.csv", index=False)
print("saved", tcga_final.shape)

saved (359, 9)
